# Notebook 5 — Job Recommendation Engine
## AI Job Market Intelligence & Smart Recommender

**Objective:** Build a hybrid, content-based job recommendation engine (skill similarity + predicted-salary fit + experience fit + city fit) on top of the salary model from Notebook 4, and — unlike the original version of this project — actually measure whether the recommendations are any good.

**Scope:**
1. Load the full job pool (`analytics_jobs.csv`) and the trained salary model / TF-IDF vectorizer
2. Build the canonical recommendation function — the exact logic `app.py` must mirror, to avoid the train/serve skew that existed in the original version (see Section 2 note)
3. Justify the hybrid scoring weights with a small sensitivity check, not just a chosen-by-feel split
4. **Evaluate recommendation quality with a real, quantified proxy metric** — the original version never evaluated this at all, just demonstrated the code runs on one hardcoded example
5. Test against multiple, varied user profiles — including a deliberate cold-start case
6. Document remaining limitations honestly

**Reference:** `docs/02_Data_Dictionary.md`, and the Notebook 4 selection-bias discussion (Section 11) — every predicted salary shown by this engine inherits that limitation.


## 1. Import Libraries

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)
np.random.seed(42)


## 2. Load Job Pool, Model, and Vectorizer

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "dataset"
MODELS_DIR = PROJECT_ROOT / "models"

analytics_df = pd.read_csv(DATA_DIR / "analytics_jobs.csv")
salary_model = joblib.load(MODELS_DIR / "salary_model.pkl")
tfidf = joblib.load(DATA_DIR / "tfidf_vectorizer.pkl")
group_mappings = joblib.load(DATA_DIR / "group_mappings.pkl")

print("Job pool:", analytics_df.shape)
print("Salary-disclosed share of pool:", f"{analytics_df['salary_disclosed'].mean():.1%}")

analytics_df["key_skills_cleaned"] = analytics_df["key_skills_cleaned"].fillna("")
job_tfidf_matrix = tfidf.transform(analytics_df["key_skills_cleaned"])
print("Job TF-IDF matrix:", job_tfidf_matrix.shape)


Job pool: (29408, 16)
Salary-disclosed share of pool: 30.4%


Job TF-IDF matrix: (29408, 300)


**Note on artifact location:** `tfidf_vectorizer.pkl` is loaded from `dataset/`, where Notebook 3 actually saved it — not `models/`. The original project's intended folder layout listed it under both, but only one real copy should exist to avoid the two ever silently drifting out of sync; `app.py` must be pointed at this same `dataset/tfidf_vectorizer.pkl` path when rebuilt, rather than expecting a duplicate under `models/`.

**Note:** the job pool used for recommendations is the **full** cleaned dataset (~29,400 postings), not just the salary-disclosed subset used to train the model. A user should be able to be recommended a great-fit job even if that job never disclosed its salary — the engine handles that explicitly via a neutral salary-match score (Section 4), rather than silently excluding those postings.

## 3. Canonical Skill-Cleaning Function — Must Match `app.py` Exactly

**This is a direct fix for a real bug identified in the original project.** The original `app.py` used a *different* skill-cleaning regex (`[^\w\s]`, stripping `+`, `#`, `.`) than the one used to prepare training data in Notebook 3 (which preserves those characters specifically to keep tokens like `C++`, `C#`, `.NET` intact). That mismatch is a train/serve skew bug: the model learns on one kind of text and gets queried with a differently-cleaned kind of text at inference time, silently degrading real-world accuracy for exactly the technical skills most likely to matter.

**The function below is copied verbatim from Notebook 3.** When `app.py` is rebuilt, it must import or exactly replicate this function — not maintain a second, independently-evolving copy.

In [3]:
def clean_skills_text(text):
    """Canonical skill-text cleaner. Must stay byte-for-byte identical to the
    version used in Notebook 3 to prepare training data — any divergence
    reintroduces train/serve skew."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+.# ]", " ", text)
    return text


def extract_skill_set(text):
    cleaned = clean_skills_text(text)
    return set(t for t in cleaned.split() if t)


# Verify the fix is in place: these tokens must survive cleaning
for token in ["c++", "c#", ".net"]:
    assert token in clean_skills_text(f"experience with {token}"), f"{token} was stripped — regex regression!"
print("Skill-cleaning consistency check passed: C++, C#, .NET all survive cleaning.")


Skill-cleaning consistency check passed: C++, C#, .NET all survive cleaning.


## 4. Predicted Salary for a User Profile

**Fix vs. the original `app.py`:** the original built the user's feature vector but never actually set the city one-hot column — a user's stated city preference was collected in the UI, used for the *recommendation* city-match score, but silently never reached the *salary prediction* itself. Fixed here by using the same `group_mappings` (top cities, city name normalization) saved in Notebook 3, so a user's city is mapped into the model exactly the way a training row's city was.

**Documented limitation:** the user profile has no `role_group` or `industry_group` input (the UI only collects skills, experience, and city) — those one-hot columns are left at zero, which corresponds to XGBoost treating them as the reference/unknown category. This is a real approximation, not a silent gap: it means the salary prediction is conditioned on experience + city + skills, but not on an assumed industry or role, which is the correct behavior for a user who hasn't told us their target industry.

In [4]:
EXP_BUCKET_ORDER = [("Entry", 2), ("Mid", 5), ("Senior", 10)]

def exp_bucket(exp):
    if exp <= 2:
        return "Entry"
    elif exp <= 5:
        return "Mid"
    elif exp <= 10:
        return "Senior"
    else:
        return "Leadership"


def predict_salary(user_skills_cleaned, user_exp, user_city, model, tfidf, group_mappings):
    feature_names = model.feature_names_in_
    user_df = pd.DataFrame(np.zeros((1, len(feature_names))), columns=feature_names)

    if "exp_avg" in user_df.columns:
        user_df.loc[0, "exp_avg"] = user_exp
    if "exp_squared" in user_df.columns:
        user_df.loc[0, "exp_squared"] = user_exp ** 2

    bucket_col = f"exp_bucket_{exp_bucket(user_exp)}"
    if bucket_col in user_df.columns:
        user_df.loc[0, bucket_col] = 1

    # City — mapped through the SAME group definitions saved in Notebook 3,
    # so a user's city is bucketed identically to how training data was.
    mapped_city = group_mappings["city_map"].get(user_city, user_city)
    city_group = mapped_city if mapped_city in group_mappings["top_cities"] else "Other"
    city_col = f"city_group_{city_group}"
    if city_col in user_df.columns:
        user_df.loc[0, city_col] = 1

    user_tfidf_vec = tfidf.transform([user_skills_cleaned])
    tfidf_terms = tfidf.get_feature_names_out()
    for i, term in enumerate(tfidf_terms):
        if term in user_df.columns:
            user_df.loc[0, term] = user_tfidf_vec[0, i]

    pred_log_salary = model.predict(user_df)[0]
    predicted_salary = np.expm1(pred_log_salary)
    return predicted_salary, user_tfidf_vec


## 5. Hybrid Scoring Function

Four signals, combined into a single ranking score:
- **Skill match** — cosine similarity between the user's TF-IDF vector and each job's TF-IDF vector
- **Salary match** — how close a job's disclosed salary is to the user's predicted salary (exponential decay); jobs with **no disclosed salary get a neutral 0.5**, not a 0 or a 1 — an undisclosed salary is neither confirmed to match nor confirmed to mismatch, so scoring it as strongly good or bad would be fabricating information the data doesn't contain
- **Experience match** — how close a job's required experience is to the user's stated experience
- **City match** — binary, whether the job's city group matches the user's mapped city group

In [5]:
def salary_match_score(job_salary, predicted_salary):
    if pd.isna(job_salary):
        return 0.5  # neutral — undisclosed, not "bad"
    diff_ratio = abs(job_salary - predicted_salary) / predicted_salary
    return float(np.exp(-2 * diff_ratio))


WEIGHTS = {"skill": 0.50, "salary": 0.20, "experience": 0.15, "city": 0.15}


def recommend_jobs(user_skills, user_exp, user_city, top_k=10, weights=WEIGHTS, exclude_uniq_id=None):
    """Return (predicted_salary, ranked_dataframe) for a given user profile.

    Set `exclude_uniq_id` to drop a specific posting from the candidate pool —
    used in Section 7's leave-one-out evaluation so a job is never used to
    recommend itself.
    """
    user_skills_cleaned = clean_skills_text(user_skills)
    predicted_salary, user_tfidf_vec = predict_salary(
        user_skills_cleaned, user_exp, user_city, salary_model, tfidf, group_mappings
    )

    pool = analytics_df if exclude_uniq_id is None else analytics_df[analytics_df["uniq_id"] != exclude_uniq_id]
    pool_tfidf = job_tfidf_matrix if exclude_uniq_id is None else tfidf.transform(pool["key_skills_cleaned"])

    scored = pool.copy()
    scored["skill_score"] = cosine_similarity(user_tfidf_vec, pool_tfidf).flatten()

    scored["salary_score"] = scored["salary_avg"].apply(lambda s: salary_match_score(s, predicted_salary))

    max_exp = analytics_df["exp_avg"].max()
    scored["exp_score"] = np.clip(1 - abs(scored["exp_avg"] - user_exp) / max_exp, 0, 1)

    mapped_city = group_mappings["city_map"].get(user_city, user_city)
    user_city_group = mapped_city if mapped_city in group_mappings["top_cities"] else "Other"
    scored["city_score"] = (scored["city_group"] == user_city_group).astype(int)

    scored["final_score"] = (
        weights["skill"] * scored["skill_score"] +
        weights["salary"] * scored["salary_score"] +
        weights["experience"] * scored["exp_score"] +
        weights["city"] * scored["city_score"]
    )

    top_jobs = scored.sort_values("final_score", ascending=False).head(top_k)
    return predicted_salary, top_jobs


## 6. Weight Sensitivity Check — Are These Weights Defensible?

**Fix vs. the original version:** the 50/20/15/15 split was chosen with no justification and never checked against alternatives. Here, three alternative weight configurations are compared on the same query — if the top recommendations barely change across reasonable weightings, that's evidence the specific numbers aren't doing unstable, arbitrary work. If they change wildly, that's a real finding that needs a more careful choice, not something to paper over.

In [6]:
alt_weights = {
    "Default (skill-heavy)": {"skill": 0.50, "salary": 0.20, "experience": 0.15, "city": 0.15},
    "Skill-dominant": {"skill": 0.70, "salary": 0.10, "experience": 0.10, "city": 0.10},
    "Balanced": {"skill": 0.25, "salary": 0.25, "experience": 0.25, "city": 0.25},
}

probe_skills = "python sql machine learning pandas"
probe_exp = 3
probe_city = "Bengaluru"

overlap_report = {}
top_id_sets = {}

for name, w in alt_weights.items():
    _, top_jobs = recommend_jobs(probe_skills, probe_exp, probe_city, top_k=10, weights=w)
    top_id_sets[name] = set(top_jobs["uniq_id"])

baseline_set = top_id_sets["Default (skill-heavy)"]
for name, id_set in top_id_sets.items():
    overlap = len(baseline_set & id_set) / len(baseline_set)
    overlap_report[name] = f"{overlap:.0%} overlap with default top-10"

for name, result in overlap_report.items():
    print(f"{name:25s}: {result}")


Default (skill-heavy)    : 100% overlap with default top-10
Skill-dominant           : 70% overlap with default top-10
Balanced                 : 70% overlap with default top-10


**Interpretation:** even the "Balanced" (25/25/25/25) configuration retains substantial overlap with the default weighting in its top-10 — skill similarity dominates the ranking under most reasonable weight choices because it has the widest spread across the candidate pool (salary/experience/city scores cluster more tightly). This is evidence the default weights are a reasonable, non-arbitrary choice, not proof they're optimal — true optimization would require real user click/conversion data, which doesn't exist for this project (documented in Section 9).

## 7. Evaluation — A Real, Quantified Proxy Metric

**This section did not exist at all in the original version of this project.** There is no real user feedback data (clicks, applications, ratings) to evaluate against — that's a genuine constraint of a portfolio project, not something to hide. Instead, a defensible **self-consistency proxy** is used: for a sample of real postings, treat that posting's own (skills, experience, city) as a synthetic user query, exclude that exact posting from the candidate pool, and check whether the top-K recommendations tend to share the same `role_group` and `industry_group` as the source posting — far more often than the pool's base rate would predict by chance.

**Why this is a meaningful check, and what it doesn't prove:** if the engine is capturing real skill-based similarity, a "Software Developer" posting's nearest neighbors by skill should skew Tech/IT-Software far more than the pool's overall Tech/IT-Software share. This validates that the similarity mechanism is coherent — it does **not** validate that users would actually prefer these recommendations, which only real usage data could show.

In [7]:
eval_sample = analytics_df[
    analytics_df["salary_disclosed"] == 1
].sample(n=120, random_state=42)

role_base_rate = analytics_df["role_group"].value_counts(normalize=True)
industry_base_rate = analytics_df["industry_group"].value_counts(normalize=True)

role_hits, industry_hits = [], []
role_base, industry_base = [], []

for _, row in eval_sample.iterrows():
    query_skills = row["key_skills_cleaned"] if row["key_skills_cleaned"] else "general"
    query_exp = row["exp_avg"] if pd.notna(row["exp_avg"]) else 3.0
    query_city = row["city_cleaned"] if pd.notna(row["city_cleaned"]) else "Other"

    _, top_jobs = recommend_jobs(
        query_skills, query_exp, query_city,
        top_k=10, exclude_uniq_id=row["uniq_id"]
    )

    role_match_rate = (top_jobs["role_group"] == row["role_group"]).mean()
    industry_match_rate = (top_jobs["industry_group"] == row["industry_group"]).mean()

    role_hits.append(role_match_rate)
    industry_hits.append(industry_match_rate)
    role_base.append(role_base_rate.get(row["role_group"], 0))
    industry_base.append(industry_base_rate.get(row["industry_group"], 0))

print(f"Evaluated on {len(eval_sample)} held-out postings (leave-one-out)")
print()
print(f"Mean role_group match rate @ 10  : {np.mean(role_hits):.1%}  "
      f"(pool base rate for these roles: {np.mean(role_base):.1%})")
print(f"Mean industry_group match rate @ 10: {np.mean(industry_hits):.1%}  "
      f"(pool base rate for these industries: {np.mean(industry_base):.1%})")


Evaluated on 120 held-out postings (leave-one-out)

Mean role_group match rate @ 10  : 56.6%  (pool base rate for these roles: 26.7%)
Mean industry_group match rate @ 10: 43.6%  (pool base rate for these industries: 20.5%)


**Reading this result:** if the match rates are meaningfully higher than the base rates, the recommender is doing real, non-random work — recommending jobs that are substantively similar to a posting's actual category, using nothing but skill text, salary fit, experience fit, and city. If they were roughly equal to the base rate, that would be evidence the skill-matching isn't adding value over recommending by base popularity, and would need to be reported as a genuine limitation rather than smoothed over.

## 8. Testing Against Multiple, Varied User Profiles

**Fix vs. the original version:** the original tested exactly one hardcoded profile. Four varied profiles are tested here, including a deliberate cold-start case, so the engine's behavior across realistic diversity — not just one lucky example — is visible.

In [8]:
test_profiles = [
    {
        "label": "Mid-level tech profile",
        "skills": "python sql machine learning pandas",
        "exp": 3,
        "city": "Bengaluru",
    },
    {
        "label": "Non-tech: sales/business development",
        "skills": "sales business development crm negotiation client relationship",
        "exp": 4,
        "city": "Mumbai",
    },
    {
        "label": "Senior leadership profile",
        "skills": "strategy leadership p&l management team building stakeholder management",
        "exp": 14,
        "city": "Delhi",
    },
    {
        "label": "COLD START: skills with no real market overlap",
        "skills": "xyzabc obscure123 nonexistentskill",
        "exp": 2,
        "city": "Pune",
    },
]

for profile in test_profiles:
    predicted_salary, top_jobs = recommend_jobs(profile["skills"], profile["exp"], profile["city"], top_k=5)

    print("=" * 90)
    print(f"PROFILE: {profile['label']}")
    print(f"  Skills: {profile['skills']} | Experience: {profile['exp']} yrs | City: {profile['city']}")
    print(f"  Predicted salary: Rs. {predicted_salary:,.0f}/year")

    max_skill_score = top_jobs["skill_score"].max()
    if max_skill_score < 0.05:
        print(f"  COLD-START WARNING: best skill match is only {max_skill_score:.3f} — these skills have "
              f"essentially no overlap with the job pool's vocabulary. Recommendations below are NOT "
              f"meaningful skill matches and should not be presented to a real user as confident matches.")

    print()
    print(top_jobs[["job_title", "city_group", "role_group", "skill_score", "final_score"]].to_string(index=False))
    print()


PROFILE: Mid-level tech profile
  Skills: python sql machine learning pandas | Experience: 3 yrs | City: Bengaluru
  Predicted salary: Rs. 497,515/year

                                          job_title city_group role_group  skill_score  final_score
  Software Senior Principal Engineer Data Scientist  Bengaluru Operations     1.000000     0.850000
                            Artificial Intelligence  Bengaluru       Tech     0.884219     0.839610
                                AI Engineer - Audio  Bengaluru       Tech     0.884219     0.829610
 Backend Java Developer - Core Java/ JSP/ Hibernate  Bengaluru       Tech     0.785776     0.787888
                                Lead Data Scientist  Bengaluru      Other     0.811607     0.778303



PROFILE: Non-tech: sales/business development
  Skills: sales business development crm negotiation client relationship | Experience: 4 yrs | City: Mumbai
  Predicted salary: Rs. 537,239/year

                                                       job_title city_group role_group  skill_score  final_score
                                                 Sales Executive     Mumbai      Sales     0.750816     0.767908
   Opening For Client Relationship Manager @kotak Life in Mumbai     Mumbai    Finance     0.587505     0.765362
 Opening For Customer Relationship Manager @kotak Life in Mumbai     Mumbai    Finance     0.507723     0.725471
     Business Development Manager - Car Parking/ Lifts Elevators     Mumbai      Sales     0.461001     0.721592
    Urgent: Contracts Specialist (Sr. Associate)_gep_navi Mumbai     Mumbai Operations     0.500074     0.700866



PROFILE: Senior leadership profile
  Skills: strategy leadership p&l management team building stakeholder management | Experience: 14 yrs | City: Delhi
  Predicted salary: Rs. 1,811,983/year

                                                             job_title city_group role_group  skill_score  final_score
                               Manager Solution Architect - .net/cloud      Delhi       Tech     0.654843     0.701542
                                    Business Transformation Consultant      Delhi      Other     0.646856     0.680928
                                                       Channel Manager      Delhi      Sales     0.654843     0.677422
                                                       Billing Manager      Delhi    Finance     0.618153     0.676577
 Associate Director Software Delivery Management / Financial / upto 30      Delhi Operations     0.424175     0.660880



PROFILE: COLD START: skills with no real market overlap
  Skills: xyzabc obscure123 nonexistentskill | Experience: 2 yrs | City: Pune
  Predicted salary: Rs. 271,637/year
  COLD-START WARNING: best skill match is only 0.000 — these skills have essentially no overlap with the job pool's vocabulary. Recommendations below are NOT meaningful skill matches and should not be presented to a real user as confident matches.

                                                            job_title city_group role_group  skill_score  final_score
                               Urgent Opening IN JAVA Developer "pune       Pune       Tech          0.0     0.495109
                                         Required Drive Test Engineer       Pune      Other          0.0     0.495109
    Software Engineer_1-2 Years Exp_Pune(Durocrete Info-tech Pvt.Ltd)       Pune       Tech          0.0     0.492609
 Hiring For Leading Bpos/for Uk /Aus \ Early Morng Shifts /call Minal       Pune      Other          0.0    

**Insight — cold start is a real, handled case, not an assumption:** the fourth profile deliberately uses skill terms absent from the job market's vocabulary. The engine still returns a ranked list (driven by salary/experience/city scores, since skill similarity is near-zero for everyone), but the explicit warning above is what the original version never had — silently returning a confidently-ranked top-10 for a query the engine has no real skill signal for would be misleading to a user. This warning must be surfaced in `app.py`'s UI, not just printed in a notebook.

## 9. Skill Gap Analysis (Explainability Feature)

In [9]:
def skill_gap(user_skills, job_skills_cleaned):
    user_set = extract_skill_set(user_skills)
    job_set = extract_skill_set(str(job_skills_cleaned))
    missing = job_set - user_set
    return missing


example_profile = test_profiles[0]
_, top_jobs = recommend_jobs(example_profile["skills"], example_profile["exp"], example_profile["city"], top_k=3)

for _, row in top_jobs.iterrows():
    missing = skill_gap(example_profile["skills"], row["key_skills_cleaned"])
    print(f"{row['job_title']}: {sorted(list(missing))[:8]}")


 Software Senior Principal Engineer Data Scientist: ['cassandra', 'hadoop', 'hbase', 'hive', 'nosql', 'pig', 'sqoop']
 Artificial Intelligence: ['architecting', 'artificial', 'intelligence']
 AI Engineer - Audio: ['algorithms', 'artificial', 'gaming', 'intelligence', 'networks', 'neural']


**Kept from the original version — this was a genuinely good feature.** Showing a user which skills separate them from a recommended job is real product value (not just a model output), and it comes for free from the same TF-IDF vocabulary already in use. No changes needed here beyond ensuring it uses the same canonical `clean_skills_text`/`extract_skill_set` functions as everything else in this notebook.

## Summary

**What changed vs. the original version of this notebook:**

| Issue | Fix |
|---|---|
| Single hardcoded test profile | Four varied profiles tested, including a deliberate cold-start case |
| No evaluation of recommendation quality at all | Leave-one-out proxy evaluation added: role/industry match rate @ 10 vs. pool base rate |
| Hybrid weights (50/20/15/15) chosen with no justification | Sensitivity check against two alternative weightings, showing top-10 overlap is reasonably stable |
| Cold start not handled | Detected explicitly (near-zero max skill score) and surfaced as a warning, not silently ranked as if meaningful |
| User's city preference collected but never reached the salary model (original `app.py` bug) | Fixed here — city is mapped through `group_mappings` and set in the model's feature vector before prediction |
| Skill-cleaning function risked diverging from Notebook 3's version (train/serve skew) | Copied verbatim, with an explicit assertion test confirming C++/C#/.NET survive cleaning |

**Known limitations, stated plainly:**
- The proxy evaluation in Section 7 validates *internal coherence* (does the engine group similar jobs together), not *user preference* — no substitute exists for real usage data, which this project does not have.
- Hybrid weights are sensitivity-checked, not learned or truly optimized — genuine optimization needs real user feedback (clicks/applications), which is documented future work, not something faked here.
- Every predicted salary shown by this engine inherits the Notebook 4 selection-bias limitation: it describes the disclosed-salary population, not the whole market.
- User `role_group`/`industry_group` are unknown at prediction time (Section 4) — a real, documented approximation in the salary prediction for every user query.

**Next stage:** `app.py` must be rebuilt to import/mirror the canonical functions defined here (`clean_skills_text`, `predict_salary`, `recommend_jobs`) exactly — not maintain independent, divergent copies, which is what caused the original train/serve skew bug in the first place.
